**Importing Libraries**

In [1]:
import torch
import torch.nn as nn

**Set fixed random seed**

In [2]:
torch.manual_seed(42)

**Custom RNN Cell**

In [3]:
class SimpleRNN(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        # Input-to-hidden weight
        self.Wx = nn.Parameter(
            torch.randn(hidden_size, input_size)
        )

        # Hidden-to-hidden weight
        self.Wh = nn.Parameter(
            torch.randn(hidden_size, hidden_size)
        )

        # Bias
        self.b = nn.Parameter(
            torch.randn(hidden_size)
        )

    def forward(self, x):

        # Initial hidden state
        h = torch.zeros(
            self.Wh.shape[0]
        )

        hidden_states = []

        # Process each time step
        for t in range(len(x)):

            h = torch.tanh(
                self.Wx @ x[t]
                + self.Wh @ h
                + self.b
            )

            hidden_states.append(h)

        return h, hidden_states


**1. Create the RNN**

In [4]:
input_size = 1
hidden_size = 1

model = SimpleRNN(input_size, hidden_size)

**2. Input sequence**

In [5]:
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0]
])

**3. Forward pass**

In [6]:
h5, hidden_states = model(x)

**4. Scalar loss**

In [7]:
L = 0.5 * h5 ** 2

**5. Backpropagation Through Time**

In [8]:
L.backward()

**6. Display hidden states**

In [9]:
print("========== HIDDEN STATES ==========")

for t, h in enumerate(hidden_states, start=1):
    print(f"h{t} =", h.detach().numpy())


print("\n========== LOSS ==========")
print("L =", L.item())

========== HIDDEN STATES ==========
h1 = [0.5162054]
h2 = [0.7506032]
h3 = [0.8719645]
h4 = [0.9345967]
h5 = [0.9666359]

========== LOSS ==========
L = 0.46719247102737427


**7. Display parameter gradients**

In [10]:
print("\n========== PARAMETER GRADIENTS ==========")

for name, parameter in model.named_parameters():

    print(f"\n{name}")
    print(parameter.grad)


========== PARAMETER GRADIENTS ==========

Wx
tensor([[0.3214]])

Wh
tensor([[0.0602]])

b
tensor([0.0645])


**Interpretation**

The gradients printed in parameter.grad represent the sensitivity of the final scalar loss to the corresponding trainable RNN parameters. Because the parameters \(W_x\), \(W_h\), and \(b\) are shared across all time steps, their gradients contain accumulated contributions from all time steps through Backpropagation Through Time. The .backward() operation automatically applies the chain rule through the unrolled computational graph and stores the resulting gradients in each parameter's .grad attribute.